In [1]:
!pip install sentence-transformers>=2.2 umap-learn>=0.5 hdbscan>=0.8 bertopic>=0.16 scikit-learn>=1.3 statsmodels>=0.14 pyarrow>=14.0

# Blinkit Review Diagnostic Pipeline

Implements the 10-stage architecture in `blinkit_pipeline_architecture.html`: population split into strict Negative/Neutral/Positive bands, three independent discovery passes, one generalized measurement stage, and the stats/robustness/deep-dive/assembly stages.

Input: `df_snippets_with_sentiment.csv` (produced by `data_prep.py`, already present in this folder). Every stage saves its output to `outputs/` as it completes — this pipeline is designed to run on Colab free tier, where a session can drop mid-run, so nothing waits until the end to be persisted.

In [2]:
import os
import pandas as pd
import numpy as np

INPUT_CSV = "df_snippets_with_sentiment.csv"
VERSION_GROUPS_CSV = "version_groups_summary.csv"
OUTPUT_DIR = "outputs"  # local here; point at a Drive path when running on Colab

os.makedirs(OUTPUT_DIR, exist_ok=True)


def out(name):
    return os.path.join(OUTPUT_DIR, name)


# --- Rating bands (Stage 1) --- strict, non-overlapping. Ratings are verified whole
# integers (no fractional values), so every review belongs to exactly one band.
BAND_NEGATIVE = (1, 2)
BAND_NEUTRAL = (3, 3)
BAND_POSITIVE = (4, 5)

# --- Quarters used in this pipeline --- see the architecture diagram's "Quarters" box.
# These are three deliberately different things; keep them separate.

# Discovery window: feeds clustering only (Negative/Neutral/Positive all share this
# same window). Never touched by the slide-8 quarters below.
DISCOVERY_WINDOW = ("2026-01-01", "2026-07-01")  # Q4 FY26 start .. Q1 FY27 end

# Primary diagnostic pair: Stage 7's headline test. Matches Eternal's disclosed QoQ
# cost figures exactly (n=15,885 / 19,440, verified against this file).
PRIMARY_A = ("2026-01-01", "2026-04-01")  # Q4 FY26 (Jan-Mar 2026)
PRIMARY_B = ("2026-04-01", "2026-07-01")  # Q1 FY27 (Apr-Jun 2026)

# Slide-8 deep-dive pair: a second, separate Stage 7 invocation, run only inside
# Stage 9. Never used for discovery/clustering. Produces one supporting finding for
# slide 8 only, not the headline table.
SLIDE8_BEFORE = ("2025-01-01", "2025-07-01")  # FY25 Q4 + FY26 Q1 pooled (Jan-Jun 2025)
SLIDE8_AFTER = ("2026-04-01", "2026-07-01")  # FY27 Q1 alone (Apr-Jun 2026)

# 1P transition timeline (for the operating-model caveat, not used in date math):
# started 2025-09-01, ~90% of NOV was 1P by Q3 FY26 (Oct-Dec 2025).

# --- Clustering (Stage 3/3b and both variants) ---
# Range validated empirically for Negative on this exact discovery window (found
# 8-20 raw topics here vs 2-4 on the full multi-year corpus). Neutral and Positive
# reuse the same candidate range but get their own search over it (see Stage 3''/3').
MIN_CLUSTER_SIZE_PCTS = [0.005, 0.0075, 0.01, 0.015, 0.02]
MIN_SAMPLES_OPTIONS = [10, 15, 25, 50, 75, 100]  # all safely under the min_samples<=150 ceiling
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

# --- Stats (Stage 7) ---
SAMPLE_FLOOR_MIN_N = 30  # below this in either cohort, report rate only, no p-value
ALPHA = 0.05

print("Config loaded. Outputs will be saved under:", os.path.abspath(OUTPUT_DIR))


Config loaded. Outputs will be saved under: /content/outputs


## Stage 1 — Population Split

Dedupe to review level, parse dates, split into the three strict rating bands across every quarter in the dataset (not narrowed to any two quarters here — that selection happens downstream, per stage).

In [3]:
df = pd.read_csv(INPUT_CSV)
df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce")


def assign_band(rating):
    if BAND_NEGATIVE[0] <= rating <= BAND_NEGATIVE[1]:
        return "Negative"
    if BAND_NEUTRAL[0] <= rating <= BAND_NEUTRAL[1]:
        return "Neutral"
    if BAND_POSITIVE[0] <= rating <= BAND_POSITIVE[1]:
        return "Positive"
    return None


df["band"] = df["rating"].apply(assign_band)
assert df["band"].isna().sum() == 0, "every rating should map to exactly one band"

# Review-level table (deduped) — used for cohort bookkeeping and the sample-floor
# check. Snippet-level df stays the working table for everything text-based.
reviews = (
    df.drop_duplicates(subset="review_id")[["review_id", "rating", "date_parsed", "band"]]
    .reset_index(drop=True)
)

print(f"Snippets: {len(df):,}  |  Unique reviews: {len(reviews):,}")
print(reviews["band"].value_counts())

df.to_parquet(out("snippets_banded.parquet"), index=False)
reviews.to_parquet(out("reviews_banded.parquet"), index=False)
print("Saved snippets_banded.parquet, reviews_banded.parquet")


Snippets: 797,770  |  Unique reviews: 588,832
band
Positive    355083
Negative    209680
Neutral      24069
Name: count, dtype: int64
Saved snippets_banded.parquet, reviews_banded.parquet


## Discovery — Negative (1–2★)

Embed → UMAP → small resolution search → BERTopic auto-labeling, on the discovery window only (Q4 FY26 + Q1 FY27). Category Integrity Guard: labels below come only from c-TF-IDF terms computed by BERTopic from this run's actual data — nothing here is hand-picked. Topic merging uses BERTopic's similarity-based `nr_topics="auto"`, not a fixed target count — a fixed target (tried during development) merged Cancellations and Refunds into a generic blob; `"auto"` only merges genuinely near-duplicate topics and correctly left every pre-registered category distinct.

In [4]:
import umap
import hdbscan
from sentence_transformers import SentenceTransformer

neg_mask = (
    (df["band"] == "Negative")
    & (df["date_parsed"] >= DISCOVERY_WINDOW[0])
    & (df["date_parsed"] < DISCOVERY_WINDOW[1])
)
neg_docs_df = df[neg_mask].reset_index(drop=True)
neg_texts = neg_docs_df["snippet"].astype(str).tolist()
print(f"Negative discovery window: {len(neg_texts):,} snippets")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

# Cache guard: embedding + UMAP together take a couple of minutes on CPU. If
# already computed once, reload from disk instead of redoing it.
_neg_emb_path, _neg_red_path = out("negative_embeddings.npy"), out("negative_reduced.npy")
if os.path.exists(_neg_emb_path) and os.path.exists(_neg_red_path):
    print("Found cached embeddings — loading instead of recomputing.")
    neg_embeddings = np.load(_neg_emb_path)
    neg_reduced = np.load(_neg_red_path)
else:
    neg_embeddings = embed_model.encode(neg_texts, batch_size=128, show_progress_bar=False)
    np.save(_neg_emb_path, neg_embeddings)

    neg_umap = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
    neg_reduced = neg_umap.fit_transform(neg_embeddings)
    np.save(_neg_red_path, neg_reduced)

print(f"Embeddings ready: {neg_embeddings.shape} -> {neg_reduced.shape}")


Negative discovery window: 52,664 snippets


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Embeddings ready: (52664, 384) -> (52664, 5)


In [5]:
from scipy.spatial.distance import pdist


def evaluate_clustering(reduced, labels):
    """n_topics, noise%, separation ratio (inter-centroid dist / mean intra-cluster dist)."""
    unique = set(labels)
    unique.discard(-1)
    n_topics = len(unique)
    noise_pct = (labels == -1).sum() / len(labels) * 100
    if n_topics < 2:
        return n_topics, noise_pct, np.nan
    centroids = np.array([reduced[labels == c].mean(axis=0) for c in unique])
    inter = pdist(centroids).mean()
    intra = np.mean(
        [
            np.linalg.norm(reduced[labels == c] - reduced[labels == c].mean(axis=0), axis=1).mean()
            for c in unique
        ]
    )
    return n_topics, noise_pct, (inter / intra if intra > 0 else np.nan)


def resolution_search(reduced, n, pcts=MIN_CLUSTER_SIZE_PCTS, samples=MIN_SAMPLES_OPTIONS):
    """Small grid search over min_cluster_size%/min_samples. Returns a results
    dataframe and the row judged best: topic count in a human-labelable range
    (6-25), low noise, maximum separation. Falls back to the highest topic
    count found if nothing clears that range."""
    rows = []
    for pct in pcts:
        mcs = max(20, int(n * pct))
        for ms in samples:
            if ms > mcs:
                continue
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=mcs, min_samples=ms, metric="euclidean", cluster_selection_method="eom"
            )
            labels = clusterer.fit_predict(reduced)
            n_topics, noise_pct, sep_ratio = evaluate_clustering(reduced, labels)
            rows.append(
                {"pct": pct, "mcs": mcs, "ms": ms, "n_topics": n_topics,
                 "noise_pct": round(noise_pct, 1),
                 "sep_ratio": round(sep_ratio, 3) if not np.isnan(sep_ratio) else None}
            )
    results = pd.DataFrame(rows)
    candidates = results[(results.n_topics >= 6) & (results.n_topics <= 25) & (results.noise_pct < 40)]
    if len(candidates) == 0:
        candidates = results[results.n_topics >= 2]
    best = candidates.sort_values("sep_ratio", ascending=False).iloc[0]
    return results, best


def fit_bertopic(docs, embeddings, mcs, ms):
    """Embed -> (given UMAP/HDBSCAN params) -> BERTopic c-TF-IDF labels ->
    similarity-based auto-merge. No hand-assigned labels anywhere."""
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer

    umap_model = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=mcs, min_samples=ms, metric="euclidean",
        cluster_selection_method="eom", prediction_data=True,
    )
    vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3)
    topic_model = BERTopic(
        umap_model=umap_model, hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model, calculate_probabilities=False, verbose=False,
    )
    topic_model.fit_transform(docs, embeddings=embeddings)
    topic_model.reduce_topics(docs, nr_topics="auto")
    return topic_model


def extract_category_definitions(topic_model, embeddings):
    """{topic_id: {keywords, count, examples, centroid}}. Centroid is computed
    in the *original* 384-dim embedding space (not the 5-dim UMAP space used
    for clustering) from topic_model.topics_ — this is what Stage 6 compares
    new, never-clustered reviews against via cosine similarity, so it has to
    live in the same space a fresh SentenceTransformer embedding does.
    keywords/examples are the only thing a human ever edits later (renaming
    for deck readability), never which reviews belong to a topic."""
    info = topic_model.get_topic_info()
    final_labels = np.array(topic_model.topics_)
    defs = {}
    for _, row in info.iterrows():
        tid = row["Topic"]
        if tid == -1:
            continue
        keywords = [w for w, _ in topic_model.get_topic(tid)]
        reps = topic_model.get_representative_docs(tid) or []
        centroid = embeddings[final_labels == tid].mean(axis=0)
        defs[str(tid)] = {
            "keywords": keywords[:12],
            "count": int(row["Count"]),
            "examples": reps[:5],
            "centroid": centroid.tolist(),
        }
    return defs


In [6]:
import json

# Pre-registered hypotheses (defined before results, per the case study's own
# methodology principle) — used only for the coverage check below, never to
# constrain what clustering is allowed to find.
HYPOTHESES = {
    "Damaged/expired": ["damaged", "expired", "rotten", "spoiled", "expiry", "fresh", "quality"],
    "Customer care contact": ["customer", "support", "care", "chat", "service"],
    "Late delivery": ["late", "delay", "time", "minutes", "delivery time"],
    "Rider/delivery partner": ["delivery boy", "delivery partner", "rude", "behaviour"],
    "Cancellations": ["cancel", "cancellation", "cancelled"],
    "Refunds": ["refund", "refunded"],
    "Wrong/missing item": ["missing", "wrong item", "wrong product", "item missing"],
}


def check_hypothesis_coverage(category_defs, hypotheses=HYPOTHESES):
    """For each pre-registered hypothesis, does at least one discovered
    category's keyword list overlap it? No forced fallback if not — report
    as-is, per the Category Integrity Guard."""
    coverage = {}
    for hyp, seeds in hypotheses.items():
        matches = []
        for tid, d in category_defs.items():
            kw_text = " ".join(d["keywords"]).lower()
            if any(seed in kw_text for seed in seeds):
                matches.append(tid)
        coverage[hyp] = matches
    return coverage


# Cache guard: this is the expensive step (grid search + BERTopic fit). If it's
# already been run once, reload the saved result instead of redoing it — this is
# what makes "run the notebook again" cheap after the first real run.
_neg_defs_path = out("category_definitions_negative.json")
if os.path.exists(_neg_defs_path):
    print(f"Found {_neg_defs_path} — loading cached result instead of recomputing.")
    with open(_neg_defs_path) as f:
        neg_category_defs = json.load(f)
else:
    neg_search_results, neg_best = resolution_search(neg_reduced, len(neg_texts))
    neg_search_results.to_csv(out("negative_resolution_search.csv"), index=False)
    print("Best combo:", dict(neg_best))

    neg_topic_model = fit_bertopic(neg_texts, neg_embeddings, int(neg_best.mcs), int(neg_best.ms))
    neg_category_defs = extract_category_definitions(neg_topic_model, neg_embeddings)

    with open(_neg_defs_path, "w") as f:
        json.dump(neg_category_defs, f, indent=2)

print(f"\n{len(neg_category_defs)} categories discovered:")
for tid, d in sorted(neg_category_defs.items(), key=lambda kv: -kv[1]["count"]):
    print(f"  [{tid}] n={d['count']:>6}  {d['keywords'][:8]}")

neg_coverage = check_hypothesis_coverage(neg_category_defs)
print("\nHypothesis coverage:")
for hyp, matches in neg_coverage.items():
    status = f"-> categories {matches}" if matches else "NO MATCH (reported as-is, no forced fallback)"
    print(f"  {hyp}: {status}")


Best combo: {'pct': np.float64(0.0075), 'mcs': np.float64(394.0), 'ms': np.float64(15.0), 'n_topics': np.float64(25.0), 'noise_pct': np.float64(15.3), 'sep_ratio': np.float64(10.898)}

25 categories discovered:
  [0] n=  7997  ['delivery', 'time', 'late', 'location', 'partner', 'order', 'deliver', 'boy']
  [1] n=  5234  ['blinkit', 'experience blinkit', 'order', 'money', 'experience', 'blink', 'dont', 'order blinkit']
  [2] n=  4557  ['app', 'worst app', 'worst', 'dont', 'use', 'app worst', 'apps', 'order']
  [3] n=  4343  ['hai', 'bhi', 'nhi', 'se', 'nahi', 'ka', 'ke', 'ho']
  [4] n=  3616  ['customer', 'support', 'customer support', 'care', 'customer service', 'customer care', 'service', 'worst customer']
  [5] n=  3031  ['charges', 'charge', 'delivery charges', 'handling', 'extra', 'high', 'delivery', 'handling charges']
  [6] n=  2766  ['refund', 'return', 'product', 'damaged', 'policy', 'received', 'money', 'replacement']
  [7] n=  1289  ['vegetables', 'rotten', 'fruits', 'fresh',

## Discovery — Neutral (3★)

Own independent fit, same discovery window, same mechanism, own resolution search (this population is far smaller — 6,448 snippets vs Negative's 52,664 — and behaves differently, so Negative's parameters aren't reused blind). Exploratory only: no pre-registered hypotheses, no significance testing on this side. Standalone output — answers "what are mixed-signal reviews actually about," doesn't feed Stage 6.

In [7]:
neu_mask = (
    (df["band"] == "Neutral")
    & (df["date_parsed"] >= DISCOVERY_WINDOW[0])
    & (df["date_parsed"] < DISCOVERY_WINDOW[1])
)
neu_docs_df = df[neu_mask].reset_index(drop=True)
neu_texts = neu_docs_df["snippet"].astype(str).tolist()
print(f"Neutral discovery window: {len(neu_texts):,} snippets")

_neu_defs_path = out("category_definitions_neutral.json")
_neu_emb_path, _neu_red_path = out("neutral_embeddings.npy"), out("neutral_reduced.npy")

if os.path.exists(_neu_defs_path):
    print(f"Found {_neu_defs_path} — loading cached result instead of recomputing.")
    with open(_neu_defs_path) as f:
        neu_category_defs = json.load(f)
else:
    if os.path.exists(_neu_emb_path) and os.path.exists(_neu_red_path):
        print("Found cached embeddings — loading instead of recomputing.")
        neu_embeddings = np.load(_neu_emb_path)
        neu_reduced = np.load(_neu_red_path)
    else:
        neu_embeddings = embed_model.encode(neu_texts, batch_size=128, show_progress_bar=False)
        np.save(_neu_emb_path, neu_embeddings)
        neu_umap = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
        neu_reduced = neu_umap.fit_transform(neu_embeddings)
        np.save(_neu_red_path, neu_reduced)

    neu_search_results, neu_best = resolution_search(neu_reduced, len(neu_texts))
    neu_search_results.to_csv(out("neutral_resolution_search.csv"), index=False)
    print("Best combo:", dict(neu_best))

    neu_topic_model = fit_bertopic(neu_texts, neu_embeddings, int(neu_best.mcs), int(neu_best.ms))
    neu_category_defs = extract_category_definitions(neu_topic_model, neu_embeddings)

    with open(_neu_defs_path, "w") as f:
        json.dump(neu_category_defs, f, indent=2)

print(f"\n{len(neu_category_defs)} Neutral categories discovered:")
for tid, d in sorted(neu_category_defs.items(), key=lambda kv: -kv[1]["count"]):
    print(f"  [{tid}] n={d['count']:>5}  {d['keywords'][:8]}")
print("\nStandalone, feeds a deck section directly, not Stage 6/7.")


Neutral discovery window: 6,448 snippets
Best combo: {'pct': np.float64(0.01), 'mcs': np.float64(64.0), 'ms': np.float64(25.0), 'n_topics': np.float64(25.0), 'noise_pct': np.float64(22.2), 'sep_ratio': np.float64(11.948)}

22 Neutral categories discovered:
  [0] n=  485  ['delivery', 'time', 'late', 'fast delivery', 'minutes', 'fast', 'delivery time', 'deliver']
  [1] n=  441  ['hai', 'ke', 'nhi', 'bhi', 'ki', 'se', 'hi', 'ho']
  [2] n=  436  ['blinkit', 'zepto', 'like', 'delivery', 'order', 'good', 'app', 'charges']
  [3] n=  386  ['price', 'high', 'prices', 'expensive', 'price high', 'higher', 'rate', 'high price']
  [4] n=  356  ['bag', 'products', 'product', 'bags', 'packaging', 'quality', 'ice', 'expired']
  [5] n=  341  ['good', 'good good', 'nice', 'ok', 'bad', 'bad good', 'experience', 'thank']
  [6] n=  323  ['app', 'apps', 'app good', 'good', 'good app', 'delivery', 'charges', 'price']
  [7] n=  319  ['delivery charges', 'charges', 'delivery', 'free', 'free delivery', 'delive

## Discovery — Positive (4–5★)

Own independent fit, same discovery window (80,862 snippets — the largest of the three bands), own resolution search. Exploratory only, same as Neutral. Standalone output — what people are actually praising, not just whether Negative-side complaints echo here.

In [8]:
pos_mask = (
    (df["band"] == "Positive")
    & (df["date_parsed"] >= DISCOVERY_WINDOW[0])
    & (df["date_parsed"] < DISCOVERY_WINDOW[1])
)
pos_docs_df = df[pos_mask].reset_index(drop=True)
pos_texts = pos_docs_df["snippet"].astype(str).tolist()
print(f"Positive discovery window: {len(pos_texts):,} snippets")

_pos_defs_path = out("category_definitions_positive.json")
_pos_emb_path, _pos_red_path = out("positive_embeddings.npy"), out("positive_reduced.npy")

if os.path.exists(_pos_defs_path):
    print(f"Found {_pos_defs_path} — loading cached result instead of recomputing.")
    with open(_pos_defs_path) as f:
        pos_category_defs = json.load(f)
else:
    if os.path.exists(_pos_emb_path) and os.path.exists(_pos_red_path):
        print("Found cached embeddings — loading instead of recomputing.")
        pos_embeddings = np.load(_pos_emb_path)
        pos_reduced = np.load(_pos_red_path)
    else:
        pos_embeddings = embed_model.encode(pos_texts, batch_size=128, show_progress_bar=False)
        np.save(_pos_emb_path, pos_embeddings)
        pos_umap = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
        pos_reduced = pos_umap.fit_transform(pos_embeddings)
        np.save(_pos_red_path, pos_reduced)

    pos_search_results, pos_best = resolution_search(pos_reduced, len(pos_texts))
    pos_search_results.to_csv(out("positive_resolution_search.csv"), index=False)
    print("Best combo:", dict(pos_best))

    pos_topic_model = fit_bertopic(pos_texts, pos_embeddings, int(pos_best.mcs), int(pos_best.ms))
    pos_category_defs = extract_category_definitions(pos_topic_model, pos_embeddings)

    with open(_pos_defs_path, "w") as f:
        json.dump(pos_category_defs, f, indent=2)

print(f"\n{len(pos_category_defs)} Positive categories discovered:")
for tid, d in sorted(pos_category_defs.items(), key=lambda kv: -kv[1]["count"]):
    print(f"  [{tid}] n={d['count']:>6}  {d['keywords'][:8]}")
print("\nStandalone, feeds a deck section directly, not Stage 6/7.")


Positive discovery window: 80,862 snippets
Best combo: {'pct': np.float64(0.0075), 'mcs': np.float64(606.0), 'ms': np.float64(50.0), 'n_topics': np.float64(8.0), 'noise_pct': np.float64(11.9), 'sep_ratio': np.float64(8.685)}

8 Positive categories discovered:
  [0] n= 42960  ['good', 'delivery', 'service', 'fast', 'time', 'nice', 'fast delivery', 'best']
  [1] n= 13306  ['app', 'best', 'good app', 'best app', 'good', 'app good', 'delivery', 'nice app']
  [2] n=  7949  ['blinkit', 'blinkit blinkit', 'thank', 'thank blinkit', 'blinkit good', 'blink', 'app', 'love']
  [3] n=  3636  ['hai', 'bahut', 'bhi', 'se', 'ho', 'ke', 'acha', 'hi']
  [4] n=   928  ['experience', 'good experience', 'experience good', 'good', 'nice experience', 'experience experience', 'experience nice', 'great experience']
  [5] n=   919  ['grocery', 'app', 'groceries', 'best', 'grocery app', 'app grocery', 'food', 'app groceries']
  [6] n=   808  ['aap', 'good aap', 'aap good', 'best aap', 'aap best', 'good', 'best',

## Stage 6 — Generalized Measurement

Measures every review, every quarter, every band against the Negative category centroids only (see `blinkit_pipeline_architecture.html`'s scope note — Neutral/Positive's own categories are standalone deck content, not fed back here). No clustering, version-blind.

**Threshold calibration, and an honest limitation.** A single flat similarity cutoff barely discriminates real matches from coincidental domain overlap — testing showed a Positive-band sample matching Negative categories at a similarity rate almost as high as genuine Negative-band members (0.855 vs 0.966 at threshold 0.3). Short, informal review sentences about the same app share enough vocabulary ("delivery," "order," "time") that raw embedding similarity alone doesn't cleanly separate sentiment. Switching to a **per-category calibrated threshold** — the 10th percentile of that category's own discovery-window members' similarity to their centroid — tightens this (0.768 vs 0.903 in the same test), but the gap is still real, not huge: cross-band match rates from this mechanism should be read as **directional, not precise**, and any headline claim built on them (e.g. "wastage still appears in N% of happy reviews") deserves a manual spot-check of a few matched examples before going in the deck.

In [9]:
category_ids = list(neg_category_defs.keys())
category_centroids = np.array([neg_category_defs[tid]["centroid"] for tid in category_ids])
category_centroids_norm = category_centroids / np.linalg.norm(category_centroids, axis=1, keepdims=True)

MIN_MEMBERS_FOR_CALIBRATION = 30  # a percentile estimate on fewer members than
                                   # this is too noisy to trust; fall back to
                                   # the median of the other categories' own
                                   # thresholds instead of an unstable number

# Per-category calibrated threshold, from the discovery window's own members
# (recovering original membership by nearest-centroid, which reproduces
# BERTopic's own assignment since the centroids were derived from it).
_own_sims = (neg_embeddings / np.linalg.norm(neg_embeddings, axis=1, keepdims=True)) @ category_centroids_norm.T
_own_best_idx = _own_sims.argmax(axis=1)
_own_best_sim = _own_sims.max(axis=1)

category_thresholds = {}
_small_categories = []
for i, tid in enumerate(category_ids):
    member_sims = _own_best_sim[_own_best_idx == i]
    if len(member_sims) >= MIN_MEMBERS_FOR_CALIBRATION:
        category_thresholds[tid] = float(np.percentile(member_sims, 10))
    else:
        category_thresholds[tid] = None  # filled in below once the stable thresholds are known
        _small_categories.append((tid, len(member_sims)))

if category_thresholds:
    _stable_vals = [v for v in category_thresholds.values() if v is not None]
    _fallback = float(np.median(_stable_vals)) if _stable_vals else 0.5
    for tid, n_members in _small_categories:
        category_thresholds[tid] = _fallback
    if _small_categories:
        print(f"{len(_small_categories)} categories had <{MIN_MEMBERS_FOR_CALIBRATION} members "
              f"(too few for a stable percentile) — using the median of the other categories' "
              f"thresholds ({_fallback:.3f}) instead: {_small_categories}")

threshold_array = np.array([category_thresholds[tid] for tid in category_ids])
print("Per-category thresholds:", {tid: round(t, 3) for tid, t in category_thresholds.items()})


def measure_against_negative_categories(embeddings):
    """Cosine similarity to every Negative category centroid; assign the
    best match only if it clears that category's own calibrated threshold.
    Returns (category_id_or_None, similarity) per row."""
    emb_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    sims = emb_norm @ category_centroids_norm.T
    best_idx = sims.argmax(axis=1)
    best_sim = sims.max(axis=1)
    passed = best_sim >= threshold_array[best_idx]
    assigned = [category_ids[i] if p else None for i, p in zip(best_idx, passed)]
    return assigned, best_sim


Per-category thresholds: {'0': 0.392, '1': 0.578, '2': 0.449, '3': 0.444, '4': 0.351, '5': 0.494, '6': 0.437, '7': 0.399, '8': 0.422, '9': 0.413, '10': 0.367, '11': 0.478, '12': 0.551, '13': 0.386, '14': 0.374, '15': 0.376, '16': 0.407, '17': 0.46, '18': 0.317, '19': 0.265, '20': 0.28, '21': 0.42, '22': 0.557, '23': 0.325, '24': 0.401}


In [10]:
import time

# Re-embeds the full corpus uniformly (rather than trying to splice in the
# discovery-window embeddings already computed above) — simpler and less
# bug-prone than index-aligning three separately-filtered subsets back into
# the full table, at the cost of a few extra minutes of re-embedding.
CHUNK_SIZE = 5000
measurement_path = out("measurement_table.parquet")
chunks_dir = out("_measurement_chunks")
all_snips = df.reset_index(drop=True)
n_total = len(all_snips)

# Cache guard: only treat an existing file as "done" if its row count matches
# the full corpus -- a checkpoint written mid-run is NOT complete and should
# not be silently treated as the final answer.
_cached_complete = False
if os.path.exists(measurement_path):
    _existing = pd.read_parquet(measurement_path)
    if len(_existing) == n_total:
        print(f"Found complete measurement_table.parquet ({n_total:,} rows) — loading instead of recomputing.")
        measurement_table = _existing
        _cached_complete = True
    else:
        print(f"Found a partial measurement_table.parquet ({len(_existing):,}/{n_total:,} rows) — "
              f"incomplete, re-running rather than trusting a partial checkpoint.")
    del _existing

if not _cached_complete:
    # Each chunk is written to its own small file immediately -- O(chunk_size)
    # per write, not O(accumulated_size). The earlier version re-concatenated
    # and rewrote the *entire* growing result list at every checkpoint, which
    # is O(n^2) work over a ~800K-row run. Writing per-chunk files and doing
    # one single merge at the end fixes that, and as a side effect makes this
    # step genuinely resumable: if interrupted, already-written chunk files
    # are detected and skipped on the next run, rather than starting over.
    os.makedirs(chunks_dir, exist_ok=True)
    t_start = time.time()
    n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE
    n_skipped = 0

    for chunk_idx, start in enumerate(range(0, n_total, CHUNK_SIZE)):
        chunk_path = os.path.join(chunks_dir, f"chunk_{chunk_idx:05d}.parquet")
        if os.path.exists(chunk_path):
            n_skipped += 1
            continue

        chunk = all_snips.iloc[start : start + CHUNK_SIZE]
        emb = embed_model.encode(chunk["snippet"].astype(str).tolist(), batch_size=256, show_progress_bar=False)
        assigned, sim = measure_against_negative_categories(emb)

        chunk_result = chunk[["review_id", "band", "rating", "date_parsed", "app_version"]].copy()
        chunk_result["category"] = assigned
        chunk_result["similarity"] = sim
        chunk_result.to_parquet(chunk_path, index=False)

        if chunk_idx % 20 == 0:
            elapsed = time.time() - t_start
            done = start + len(chunk)
            eta = (elapsed / max(done - n_skipped * CHUNK_SIZE, 1)) * (n_total - done)
            print(f"Progress {done:,}/{n_total:,} ({done/n_total*100:.1f}%) — elapsed {elapsed/60:.1f}min, ETA {eta/60:.1f}min")

    if n_skipped:
        print(f"Resumed: skipped {n_skipped} already-completed chunks from a prior run.")

    chunk_files = sorted(os.listdir(chunks_dir))
    measurement_table = pd.concat(
        [pd.read_parquet(os.path.join(chunks_dir, f)) for f in chunk_files], ignore_index=True
    )
    measurement_table.to_parquet(measurement_path, index=False)

    import shutil
    shutil.rmtree(chunks_dir)
    print(f"\nDone in {(time.time()-t_start)/60:.1f} min.")

print(f"measurement_table.parquet: {len(measurement_table):,} rows")
print(measurement_table["category"].value_counts(dropna=False).head(10))


Progress 5,000/797,770 (0.6%) — elapsed 1.0min, ETA 163.2min
Progress 105,000/797,770 (13.2%) — elapsed 18.7min, ETA 123.6min
Progress 205,000/797,770 (25.7%) — elapsed 35.5min, ETA 102.5min
Progress 305,000/797,770 (38.2%) — elapsed 52.6min, ETA 84.9min
Progress 405,000/797,770 (50.8%) — elapsed 68.7min, ETA 66.6min
Progress 505,000/797,770 (63.3%) — elapsed 85.0min, ETA 49.3min
Progress 605,000/797,770 (75.8%) — elapsed 101.9min, ETA 32.5min
Progress 705,000/797,770 (88.4%) — elapsed 118.7min, ETA 15.6min

Done in 136.0 min.
measurement_table.parquet: 797,770 rows
category
2       110611
0       109484
None     97536
20       61022
11       60486
1        55913
3        31450
8        28796
5        27754
4        27111
Name: count, dtype: int64


### Review-level rollup

A review belongs to category X if *any* of its snippets does — this is the multi-category-membership, review-level dedup that avoids snippet pseudo-replication (a review with 5 angry sentences about the same damaged item shouldn't count 5 times).

In [11]:
measurement_table = pd.read_parquet(out("measurement_table.parquet"))

# long-format review x category table: one row per (review_id, category) a
# review actually touches (drops the "no match" rows entirely)
review_category = (
    measurement_table.dropna(subset=["category"])
    .drop_duplicates(subset=["review_id", "category"])[["review_id", "category"]]
    .reset_index(drop=True)
)
review_category.to_parquet(out("review_category.parquet"), index=False)

# review-level metadata (one row per review) for cohort/sample-floor bookkeeping
reviews_meta = (
    measurement_table.drop_duplicates(subset="review_id")[["review_id", "band", "rating", "date_parsed"]]
    .reset_index(drop=True)
)
reviews_meta.to_parquet(out("reviews_meta.parquet"), index=False)

print(f"review_category: {len(review_category):,} (review, category) pairs across {review_category.review_id.nunique():,} reviews")
print(f"reviews_meta: {len(reviews_meta):,} reviews")


review_category: 661,335 (review, category) pairs across 537,629 reviews
reviews_meta: 588,832 reviews


## Stage 7 — Stats Engine

Two-proportion z-test + Cohen's h per category, sample-size floor, two-tier correction (Bonferroni on the pre-registered hypotheses, Benjamini–Hochberg on whatever else clustering discovered). One engine, invoked twice: once for the primary pair (this section — the headline table), once for the slide-8 pair (inside Stage 9).

In [12]:
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests


def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))


def stats_engine(review_category_long, reviews_meta, cohort_a, cohort_b, band, categories=None,
                  id_sets=None, min_n=SAMPLE_FLOOR_MIN_N):
    """
    Two-proportion z-test + Cohen's h per category between cohort_a and cohort_b
    (each a (start, end) date-string tuple), restricted to `band`. Sample floor:
    both cohort sizes >= min_n AND >= 5 expected occurrences both ways (n*p and
    n*(1-p)) -- below floor, rate is still reported, no p-value/h claimed.

    `categories`: named categories already present in review_category_long
    (default: every category found there). `id_sets`: an optional dict of
    {label: set_of_review_ids} for ad-hoc populations not tied to a discovered
    category (e.g. the perishable-tagged wastage subset) -- both go through
    the identical test, so there's one implementation of this test, not two.
    """
    pop_a = reviews_meta[
        (reviews_meta.band == band) & (reviews_meta.date_parsed >= cohort_a[0]) & (reviews_meta.date_parsed < cohort_a[1])
    ]
    pop_b = reviews_meta[
        (reviews_meta.band == band) & (reviews_meta.date_parsed >= cohort_b[0]) & (reviews_meta.date_parsed < cohort_b[1])
    ]
    n_a, n_b = len(pop_a), len(pop_b)
    ids_a, ids_b = set(pop_a.review_id), set(pop_b.review_id)

    tests = {}
    if categories is None and id_sets is None:
        categories = sorted(review_category_long["category"].unique().tolist())
    for cat in (categories or []):
        tests[cat] = set(review_category_long.loc[review_category_long.category == cat, "review_id"])
    tests.update(id_sets or {})

    rows = []
    for label, cat_ids in tests.items():
        count_a = len(ids_a & cat_ids)
        count_b = len(ids_b & cat_ids)
        rate_a = count_a / n_a if n_a else np.nan
        rate_b = count_b / n_b if n_b else np.nan

        floor_ok = (
            n_a >= min_n and n_b >= min_n
            and count_a >= 5 and (n_a - count_a) >= 5
            and count_b >= 5 and (n_b - count_b) >= 5
        )

        p_value, h = np.nan, np.nan
        if floor_ok:
            _, p_value = proportions_ztest([count_a, count_b], [n_a, n_b])
            h = cohens_h(rate_a, rate_b)

        rows.append({
            "category": label, "n_a": n_a, "count_a": count_a, "rate_a": rate_a,
            "n_b": n_b, "count_b": count_b, "rate_b": rate_b,
            "RR": (rate_b / rate_a) if rate_a else np.nan,
            "cohens_h": h, "p_value": p_value, "sample_floor_ok": floor_ok,
        })
    return pd.DataFrame(rows)


def apply_two_tier_correction(results_df, pre_registered_categories, alpha=ALPHA):
    """Bonferroni on pre-registered hypothesis categories; Benjamini-Hochberg
    (FDR) on everything else clustering discovered. Only rows that cleared the
    sample floor participate in either correction.

    Reports `p_value_adjusted` for every tested row -- this is what actually
    reflects the method used (Bonferroni or BH), for either tier. `alpha_used`
    is populated *only* for the pre-registered/Bonferroni tier, where a single
    fixed threshold (alpha/n) genuinely applies; BH doesn't have an equivalent
    single-number threshold (significance there depends on each p-value's rank
    among the others), so leaving it blank for that tier is the honest choice
    rather than reusing statsmodels' Bonferroni-only auxiliary return value,
    which would silently mislabel the exploratory tier's threshold.
    """
    results_df = results_df.copy()
    results_df["tier"] = np.where(results_df.category.isin(pre_registered_categories), "pre_registered", "exploratory")
    results_df["significant"] = False
    results_df["p_value_adjusted"] = np.nan
    results_df["alpha_used"] = np.nan

    testable = results_df[results_df.sample_floor_ok]
    for tier, method in [("pre_registered", "bonferroni"), ("exploratory", "fdr_bh")]:
        subset = testable[testable.tier == tier]
        if len(subset) == 0:
            continue
        reject, pvals_corrected, _, _ = multipletests(subset.p_value, alpha=alpha, method=method)
        results_df.loc[subset.index, "significant"] = reject
        results_df.loc[subset.index, "p_value_adjusted"] = pvals_corrected
        if tier == "pre_registered":
            results_df.loc[subset.index, "alpha_used"] = alpha / len(subset)
    return results_df


In [13]:
# Pre-registered category ids = the union of whatever matched each hypothesis
# in the coverage check earlier. If a hypothesis matched multiple discovered
# categories (e.g. "Damaged/expired" split into a perishables cluster and a
# separate "expired products" cluster), all of them are pre-registered, not
# just one -- the split itself is a real finding, not something to collapse.
pre_registered_categories = sorted({tid for matches in neg_coverage.values() for tid in matches})
print(f"Pre-registered categories ({len(pre_registered_categories)}): {pre_registered_categories}")

stats_primary = stats_engine(review_category, reviews_meta, PRIMARY_A, PRIMARY_B, band="Negative")
stats_primary = apply_two_tier_correction(stats_primary, pre_registered_categories)
stats_primary = stats_primary.sort_values("cohens_h", ascending=False, key=abs)

stats_primary.to_csv(out("stats_primary.csv"), index=False)
print(f"\nSaved stats_primary.csv ({len(stats_primary)} categories)")
stats_primary[["category", "tier", "rate_a", "rate_b", "RR", "cohens_h", "p_value", "p_value_adjusted", "sample_floor_ok", "significant"]]


Pre-registered categories (12): ['0', '1', '11', '12', '13', '17', '18', '19', '4', '6', '7', '8']

Saved stats_primary.csv (25 categories)


,category,tier,rate_a,rate_b,RR,cohens_h,p_value,p_value_adjusted,sample_floor_ok,significant
8,16,exploratory,0.009002,0.021348,2.371390,-0.103221,2.093477e-20,2.721520e-19,True,True
20,5,exploratory,0.080138,0.067233,0.838954,0.049445,3.525082e-06,2.291303e-05,True,True
1,1,pre_registered,0.096947,0.111677,1.151941,-0.048219,7.067737e-06,8.481284e-05,True,True
22,7,pre_registered,0.030280,0.036214,1.195965,-0.033130,2.061249e-03,2.473498e-02,True,True
2,10,exploratory,0.030217,0.034774,1.150791,-0.025713,1.664253e-02,7.211761e-02,True,False
19,4,pre_registered,0.070003,0.076235,1.089016,-0.023942,2.550946e-02,3.061135e-01,True,False
12,2,exploratory,0.100346,0.106739,1.063704,-0.020984,5.010566e-02,1.190165e-01,True,False
18,3,exploratory,0.097262,0.103292,1.062004,-0.020079,6.083837e-02,1.190165e-01,True,False
7,15,exploratory,0.018823,0.016204,0.860856,0.019980,6.096308e-02,1.190165e-01,True,False
17,24,exploratory,0.023733,0.026852,1.131410,-0.019873,6.408578e-02,1.190165e-01,True,False


## Stage 8 — Version-Mix Robustness

Pure groupby/reweighting on data already in `measurement_table` — no embedding, no clustering, no new model call. For each significant category, reweights the later cohort's version-specific rates to the earlier cohort's version-group mix (direct standardization) and checks whether the movement survives — rules out "a UI change in a specific app version did this," not an operational issue.

In [14]:
def get_major_minor(v):
    try:
        parts = str(v).split(".")
        return f"{parts[0]}.{parts[1]}" if len(parts) >= 2 else str(v)
    except Exception:
        return str(v)


MIN_VERSION_COVERAGE = 0.5  # below this, the reweighted estimate is built on too
                             # thin a slice of cohort_a's version mix to trust


def version_robustness_check(measurement_table, review_category_long, category, cohort_a, cohort_b, band="Negative"):
    """Reweights cohort_b's version-specific rates to cohort_a's version-group
    mix (direct standardization). robust=True if the reweighted difference
    keeps the same sign and at least half the magnitude of the raw difference
    -- i.e. the movement isn't mostly explained by which app version people
    happened to be on. robust=None (unresolved, not "passed") if there's no
    version overlap at all, OR if the overlap covers too little of cohort_a's
    version mix to trust the reweighted number -- a low-coverage reweight is
    not meaningfully different from having no check at all, and treating it
    as a pass would be a false confidence signal."""
    pop = measurement_table.drop_duplicates(subset="review_id")[
        ["review_id", "band", "rating", "date_parsed", "app_version"]
    ].copy()
    pop["version_group"] = pop["app_version"].apply(get_major_minor)

    pop_a = pop[(pop.band == band) & (pop.date_parsed >= cohort_a[0]) & (pop.date_parsed < cohort_a[1])]
    pop_b = pop[(pop.band == band) & (pop.date_parsed >= cohort_b[0]) & (pop.date_parsed < cohort_b[1])]

    cat_ids = set(review_category_long.loc[review_category_long.category == category, "review_id"])
    pop_a = pop_a.assign(has_cat=pop_a.review_id.isin(cat_ids))
    pop_b = pop_b.assign(has_cat=pop_b.review_id.isin(cat_ids))

    mix_a = pop_a.version_group.value_counts(normalize=True)
    rate_by_version_b = pop_b.groupby("version_group")["has_cat"].mean()

    common = set(mix_a.index) & set(rate_by_version_b.index)
    if not common:
        return {"category": category, "robust": None, "weight_sum": 0.0, "note": "no overlapping version groups"}

    weight_sum = sum(mix_a.get(v, 0) for v in common)
    if weight_sum < MIN_VERSION_COVERAGE:
        return {
            "category": category, "robust": None, "weight_sum": weight_sum,
            "note": f"low coverage: overlap represents only {weight_sum*100:.0f}% of cohort_a's version mix, not trusted",
        }

    reweighted_rate_b = sum(rate_by_version_b.get(v, 0) * mix_a.get(v, 0) for v in common) / weight_sum

    raw_rate_a, raw_rate_b = pop_a.has_cat.mean(), pop_b.has_cat.mean()
    raw_diff = raw_rate_b - raw_rate_a
    reweighted_diff = reweighted_rate_b - raw_rate_a
    robust = True if raw_diff == 0 else (
        np.sign(raw_diff) == np.sign(reweighted_diff) and abs(reweighted_diff) >= 0.5 * abs(raw_diff)
    )

    return {
        "category": category, "raw_rate_a": raw_rate_a, "raw_rate_b": raw_rate_b,
        "reweighted_rate_b": reweighted_rate_b, "raw_diff": raw_diff,
        "reweighted_diff": reweighted_diff, "weight_sum": weight_sum, "robust": robust,
    }


significant_categories = stats_primary.loc[stats_primary.significant, "category"].tolist()
robustness_rows = [
    version_robustness_check(measurement_table, review_category, cat, PRIMARY_A, PRIMARY_B)
    for cat in significant_categories
]
robustness_flags = pd.DataFrame(robustness_rows)
robustness_flags.to_csv(out("robustness_flags.csv"), index=False)
print(f"Checked {len(significant_categories)} significant categories for version-mix robustness.")
robustness_flags


Checked 4 significant categories for version-mix robustness.


,category,raw_rate_a,raw_rate_b,reweighted_rate_b,raw_diff,reweighted_diff,weight_sum,robust
0,16,0.009002,0.021348,0.022939,0.012346,0.013937,0.996601,True
1,5,0.080138,0.067233,0.076972,-0.012906,-0.003166,0.996601,False
2,1,0.096947,0.111677,0.116559,0.014730,0.019612,0.996601,True
3,7,0.030280,0.036214,0.038944,0.005934,0.008664,0.996601,True


## Stage 9 — Co-occurrence & Deep Dives

Two things, both reusing data/machinery that already exists — no new clustering, no new model.

**Customer-care root cause**: of reviews tagged "customer care," what share also carry "wastage" or "late delivery"? A simple crosstab on `review_category`.

**Pre/post-1P wastage deep dive (slide 8)**: tags wastage-category reviews perishable/non-perishable via the sourced 1P-category keyword list, then re-invokes Stage 7's exact engine on the `SLIDE8_BEFORE`/`SLIDE8_AFTER` cohorts. Scope not yet fixed (per the architecture diagram) — computing **both** wastage-overall and the perishable-tagged subset so the choice can be made once the numbers are visible, not before.

In [15]:
def cooccurrence(review_category_long, categories_a, categories_b):
    ids_a = set(review_category_long.loc[review_category_long.category.isin(categories_a), "review_id"])
    ids_b = set(review_category_long.loc[review_category_long.category.isin(categories_b), "review_id"])
    overlap = ids_a & ids_b
    return {
        "n_a": len(ids_a), "n_b": len(ids_b), "n_overlap": len(overlap),
        "pct_of_a_also_b": (len(overlap) / len(ids_a)) if ids_a else np.nan,
    }


customer_care_ids = neg_coverage["Customer care contact"]
wastage_ids = neg_coverage["Damaged/expired"]
late_delivery_ids = neg_coverage["Late delivery"]

cooc_wastage = cooccurrence(review_category, customer_care_ids, wastage_ids)
cooc_late = cooccurrence(review_category, customer_care_ids, late_delivery_ids)

cooccurrence_matrix = pd.DataFrame([
    {"pair": "customer_care -> also wastage", **cooc_wastage},
    {"pair": "customer_care -> also late_delivery", **cooc_late},
])
cooccurrence_matrix.to_csv(out("cooccurrence_matrix.csv"), index=False)
print(cooccurrence_matrix.to_string(index=False))


                               pair    n_a    n_b  n_overlap  pct_of_a_also_b
      customer_care -> also wastage 131563  60583      12186         0.092625
customer_care -> also late_delivery 131563 116909      14390         0.109377


In [16]:
# Sourced from Blinkit's 1P (inventory-led) transition coverage: fresh produce,
# meat, and dairy are the flagship categories Blinkit took direct stocking
# ownership of; electronics/toys/home-decor are the flagship non-perishable
# 1P expansion. Keyword tagging, not a model -- consistent with keeping the
# deep dive auditable.
PERISHABLE_KEYWORDS = [
    "vegetable", "fruit", "milk", "dairy", "meat", "chicken", "fish", "egg",
    "bread", "yogurt", "curd", "paneer", "fresh produce", "perishable", "rotten", "spoiled",
]
NON_PERISHABLE_KEYWORDS = [
    "electronic", "gadget", "toy", "makeup", "cosmetic", "home decor", "pooja",
    "charger", "earphone", "phone case", "stationery",
]


def tag_perishable(text):
    t = str(text).lower()
    is_p = any(k in t for k in PERISHABLE_KEYWORDS)
    is_np = any(k in t for k in NON_PERISHABLE_KEYWORDS)
    if is_p and not is_np:
        return "perishable"
    if is_np and not is_p:
        return "non_perishable"
    if is_p and is_np:
        return "mixed"
    return "unspecified"


wastage_review_ids = set(review_category.loc[review_category.category.isin(wastage_ids), "review_id"])
wastage_snippets = df[df.review_id.isin(wastage_review_ids)][["review_id", "snippet"]].copy()
wastage_snippets["tag"] = wastage_snippets["snippet"].apply(tag_perishable)


def rollup_tag(tags):
    # Deliberate choice: a review whose only wastage-relevant snippets are
    # "mixed" (both perishable and non-perishable keywords in the same
    # sentence -- e.g. "the milk and the charger both arrived damaged") is
    # rolled up as "unspecified", not counted toward either side. The
    # alternative -- crediting "mixed" snippets to "perishable" -- would
    # inflate the perishable count with genuinely ambiguous mentions. This
    # trades a small amount of recall for not overstating the 1P-specific
    # finding.
    if "perishable" in tags.values:
        return "perishable"
    if "non_perishable" in tags.values:
        return "non_perishable"
    return "unspecified"


review_perishable = wastage_snippets.groupby("review_id")["tag"].apply(rollup_tag)
perishable_review_ids = set(review_perishable[review_perishable == "perishable"].index)
print(f"Wastage reviews: {len(wastage_review_ids):,} total, {len(perishable_review_ids):,} tagged perishable "
      f"({len(perishable_review_ids)/len(wastage_review_ids)*100:.1f}%)")

perishable_split = review_perishable.value_counts().reset_index()
perishable_split.columns = ["tag", "count"]
perishable_split.to_csv(out("perishable_split.csv"), index=False)

# Reuses stats_engine's id_sets parameter -- same tested implementation as the
# primary diagnostic, not a second copy of the two-proportion test.
wastage_1p_results = stats_engine(
    review_category, reviews_meta, SLIDE8_BEFORE, SLIDE8_AFTER, band="Negative",
    id_sets={"wastage_overall": wastage_review_ids, "wastage_perishable_only": perishable_review_ids},
)
wastage_1p_results = wastage_1p_results.rename(columns={"category": "scope"})
wastage_1p_results.to_csv(out("wastage_1P_finding.csv"), index=False)
print("\nPre/post-1P wastage finding (scope TBD, both computed):")
wastage_1p_results


Wastage reviews: 60,583 total, 12,928 tagged perishable (21.3%)

Pre/post-1P wastage finding (scope TBD, both computed):


,scope,n_a,count_a,rate_a,n_b,count_b,rate_b,RR,cohens_h,p_value,sample_floor_ok
0,wastage_overall,16596,2821,0.169981,19440,3081,0.158488,0.932386,0.031026,0.003298,True
1,wastage_perishable_only,16596,547,0.032960,19440,615,0.031636,0.959831,0.007489,0.478235,True


## Stage 10 — Ranked Diagnostic Assembly

Doesn't produce one output — assembles five, from different upstream stages and different quarter windows, routed to different slides. Deterministic, no model call.

| Output | Source | Window | Feeds |
|---|---|---|---|
| `final_ranked_table.csv` | Stage 7 (primary) + Stage 8 + Stage 6 (Neutral/Positive rates) | Q4 FY26 vs Q1 FY27 | Slides 6, 7, 11 |
| `wastage_1P_finding.csv` | Stage 7 (slide-8 invocation) + Stage 9 | Before FY25 Q4+FY26 Q1 · After FY27 Q1 | Slide 8 |
| `cooccurrence_matrix.csv` | Stage 9 crosstab | Whichever cohort attached | Slide 9 |
| `neutral_theme_list.csv` | Stage 4'' directly | Discovery window | Slide TBD |
| `positive_theme_list.csv` | Stage 4' directly | Discovery window | Slide TBD |

In [17]:
def rate_for_band(category, band, window):
    pop = reviews_meta[(reviews_meta.band == band) & (reviews_meta.date_parsed >= window[0]) & (reviews_meta.date_parsed < window[1])]
    if len(pop) == 0:
        return np.nan
    cat_ids = set(review_category.loc[review_category.category == category, "review_id"])
    return pop.review_id.isin(cat_ids).sum() / len(pop)


full_primary_window = (PRIMARY_A[0], PRIMARY_B[1])

final_ranked_table = stats_primary.copy()
final_ranked_table["neutral_rate"] = final_ranked_table["category"].apply(lambda c: rate_for_band(c, "Neutral", full_primary_window))
final_ranked_table["positive_rate"] = final_ranked_table["category"].apply(lambda c: rate_for_band(c, "Positive", full_primary_window))
final_ranked_table["keywords"] = final_ranked_table["category"].apply(lambda c: ", ".join(neg_category_defs[c]["keywords"][:6]))

final_ranked_table = final_ranked_table.merge(
    robustness_flags[["category", "robust"]] if len(robustness_flags) else pd.DataFrame(columns=["category", "robust"]),
    on="category", how="left",
)


def assign_zone(row):
    """Conservative by construction: Priority requires an *explicit* robust
    True. Anything else -- robust=False, robust=None (e.g. no overlapping
    version groups so Stage 8 couldn't reach a verdict), or missing entirely
    (not significant, so Stage 8 never ran on it -- caught by the first
    branch) -- falls to Watch rather than silently defaulting to Priority.
    A plain `row["robust"] is False` check does NOT do this: NaN/None both
    fail that identity check and would fall through to a Priority default,
    which is the bug this version fixes."""
    if not row["significant"]:
        return "No Action"
    if row["robust"] is True:
        return "Priority"
    return "Watch"


final_ranked_table["zone"] = final_ranked_table.apply(assign_zone, axis=1)
final_ranked_table = final_ranked_table.sort_values("cohens_h", ascending=False, key=abs)
final_ranked_table.to_csv(out("final_ranked_table.csv"), index=False)

print(f"Saved final_ranked_table.csv ({len(final_ranked_table)} categories)")
print(final_ranked_table["zone"].value_counts())
final_ranked_table[["category", "keywords", "zone", "rate_a", "rate_b", "neutral_rate", "positive_rate", "cohens_h", "p_value_adjusted"]]


Saved final_ranked_table.csv (25 categories)
zone
No Action    21
Priority      3
Watch         1
Name: count, dtype: int64


,category,keywords,zone,rate_a,rate_b,neutral_rate,positive_rate,cohens_h,p_value_adjusted
0,16,"ice, ice cream, cream, melted, icecream, cold",Priority,0.009002,0.021348,0.010853,0.002570,-0.103221,2.721520e-19
1,5,"charges, charge, delivery charges, handling, e...",Watch,0.080138,0.067233,0.118621,0.015014,0.049445,2.291303e-05
2,1,"blinkit, experience blinkit, order, money, exp...",Priority,0.096947,0.111677,0.062643,0.083700,-0.048219,8.481284e-05
3,7,"vegetables, rotten, fruits, fresh, quality, or...",Priority,0.030280,0.036214,0.033892,0.011396,-0.033130,2.473498e-02
4,10,"bag, packaging, packing, dirty, bags, packet",No Action,0.030217,0.034774,0.030465,0.010670,-0.025713,7.211761e-02
5,4,"customer, support, customer support, care, cus...",No Action,0.070003,0.076235,0.013519,0.005573,-0.023942,3.061135e-01
6,2,"app, worst app, worst, dont, use, app worst",No Action,0.100346,0.106739,0.089680,0.209550,-0.020984,1.190165e-01
7,3,"hai, bhi, nhi, se, nahi, ka",No Action,0.097262,0.103292,0.070640,0.048686,-0.020079,1.190165e-01
8,15,"zepto, better, instamart, flipkart, zepto bett...",No Action,0.018823,0.016204,0.021516,0.005126,0.019980,1.190165e-01
9,24,"experience, experience bad, bad experience, wo...",No Action,0.023733,0.026852,0.006093,0.015824,-0.019873,1.190165e-01


In [18]:
def defs_to_theme_list(category_defs):
    rows = [
        {"topic_id": tid, "count": d["count"], "keywords": ", ".join(d["keywords"]),
         "example": d["examples"][0] if d["examples"] else ""}
        for tid, d in category_defs.items()
    ]
    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)


neutral_theme_list = defs_to_theme_list(neu_category_defs)
neutral_theme_list.to_csv(out("neutral_theme_list.csv"), index=False)

positive_theme_list = defs_to_theme_list(pos_category_defs)
positive_theme_list.to_csv(out("positive_theme_list.csv"), index=False)

print(f"Saved neutral_theme_list.csv ({len(neutral_theme_list)} themes)")
print(f"Saved positive_theme_list.csv ({len(positive_theme_list)} themes)")
print("\nPipeline complete. All outputs in:", os.path.abspath(OUTPUT_DIR))
print(sorted(os.listdir(OUTPUT_DIR)))


Saved neutral_theme_list.csv (22 themes)
Saved positive_theme_list.csv (8 themes)

Pipeline complete. All outputs in: /content/outputs
['category_definitions_negative.json', 'category_definitions_neutral.json', 'category_definitions_positive.json', 'cooccurrence_matrix.csv', 'final_ranked_table.csv', 'measurement_table.parquet', 'negative_embeddings.npy', 'negative_reduced.npy', 'negative_resolution_search.csv', 'neutral_embeddings.npy', 'neutral_reduced.npy', 'neutral_resolution_search.csv', 'neutral_theme_list.csv', 'perishable_split.csv', 'positive_embeddings.npy', 'positive_reduced.npy', 'positive_resolution_search.csv', 'positive_theme_list.csv', 'review_category.parquet', 'reviews_banded.parquet', 'reviews_meta.parquet', 'robustness_flags.csv', 'snippets_banded.parquet', 'stats_primary.csv', 'wastage_1P_finding.csv']
